In [236]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from torch.utils.data.dataloader import DataLoader
from torch.utils.data.dataset import TensorDataset

plt.style.use("default")

In [237]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [238]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False) # X: (70000, 784)

In [239]:
X, y = mnist["data"], mnist["target"]
X = torch.Tensor(X.astype(np.float32) / 255.0).reshape(-1, 1, 28, 28)
y = torch.Tensor(y.astype(np.int64))
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=10000, random_state=42, stratify=y
)
X_test = X_test.to(device)

In [240]:
X_trainval.shape

torch.Size([60000, 1, 28, 28])

In [241]:
train_dataset = TensorDataset(X_trainval.to(device), y_trainval.to(device))
test_dataset = TensorDataset(X_test.to(device), y_test.to(device))

test_loader = DataLoader(test_dataset, batch_size=1000)

In [242]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()

        # Thanx to https://medium.com/data-science-collective/implementing-cnn-in-pytorch-testing-on-mnist-99-26-test-accuracy-5c63876c6ac8
        self.model = nn.Sequential(
            nn.ZeroPad2d(2),
            nn.Conv2d(1, 16, 5, 1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.LazyLinear(out_features=10),
        )

        self.train_loss_per_epoch = None
        self.val_loss_per_batch = None
        self.best_val_loss = None

    def forward(self, x):
        x = self.model(x)
        return x


In [243]:
def train_one_model(max_epochs: int, batch_size: int, lr: float) -> MLP:
    model = MLP()
    model.to(device)
    model.train()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    n_reports = 10
    print_every = max(max_epochs // n_reports, 1)

    train_loss_per_epoch = np.zeros(max_epochs)
    val_loss_per_batch = np.zeros(max_epochs)

    for epoch in range(max_epochs):
        for data, target in train_loader:
            optimizer.zero_grad()

            y_pred = model(data)

            loss = loss_fn(y_pred, target.long())
            loss.backward()

            optimizer.step()

        model.eval()

        with torch.no_grad():
            loss = 0.0

            for data, target in train_loader:
                y_pred = model(data)
                loss += loss_fn(y_pred, target.long()).item()

            loss_avg = loss / len(train_loader)
            train_loss_per_epoch[epoch] = loss_avg

            loss = 0.0

            for data, target in test_loader:
                y_pred = model(data)
                loss += loss_fn(y_pred, target.long()).item()

            loss_avg = loss / len(test_loader)
            val_loss_per_batch[epoch] = loss_avg

            # if epoch % print_every == 0:
            #     print(f"{epoch=}\tCrossEntropy = {loss_avg:.3f}")

        model.train()

    model.train_loss_per_epoch = train_loss_per_epoch
    model.val_loss_per_batch = val_loss_per_batch
    model.best_val_loss = np.min(val_loss_per_batch).item()

    model.eval()
    return model

In [ ]:
batch_sizes = [
    4096,
    2048,
    1024,
    512,
    256,
    128,
    64,
    32,
    16,
    8,
]

learning_rates = [
    0.00001,
    0.0001,
    0.001,
    0.01,
    0.1,
]

models = np.zeros((len(batch_sizes), len(learning_rates)), dtype=object)

training_durations = np.zeros_like(models, dtype=float)

for i, batch_size in enumerate(batch_sizes):
    for j, lr in enumerate(learning_rates):
        lr = 0.1

        start = time.time()
        model = train_one_model(5, batch_size, lr)
        duration = time.time() - start

        models[i, j] = model
        training_durations[i, j] = duration

        perc_done = 100*(i*models.shape[1] + j + 1)//models.size
        print(f"[{perc_done:3n}%] model ({i}, {j}) trained with best val loss = {model.best_val_loss}")

/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


[  2%] model (0, 0) trained with best val loss = 0.3803024679422379
[  4%] model (0, 1) trained with best val loss = 1.53330500125885
[  6%] model (0, 2) trained with best val loss = 0.44227359890937806
[  8%] model (0, 3) trained with best val loss = 0.4673261374235153
[ 10%] model (0, 4) trained with best val loss = 1.2728389143943786


In [ ]:
losses = np.vectorize(lambda m: m.best_val_loss)(models)
losses

In [ ]:
# Appends the results to the CSV
try:
    df = pd.read_csv("batch_size_losses.csv", header=None)
    df = pd.concat([df, pd.DataFrame(losses)], axis=0)
except FileNotFoundError:
    df = pd.DataFrame(losses)

df.to_csv("batch_size_losses.csv", index=False, header=False)
df

In [ ]:
sns.heatmap(losses, annot=True, vmax=0.45, fmt=".3f", xticklabels=learning_rates, yticklabels=batch_sizes, cmap="Blues")

In [ ]:
sns.heatmap(training_durations, annot=True, fmt=".2f", xticklabels=learning_rates, yticklabels=batch_sizes, cmap="Blues")

In [ ]:
inv_training_eff = losses*training_durations

inv_training_eff

In [ ]:
sns.heatmap(inv_training_eff, annot=True, vmax=8, fmt=".2f", xticklabels=learning_rates, yticklabels=batch_sizes, cmap="Blues")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

titles = ["Best loss", "Training duration [s]", "Inverted training efficiency"]
for ax, arr, title in zip(axs, [losses, training_durations, inv_training_eff], titles):
    sns.heatmap(arr, annot=True, fmt=".2f", xticklabels=learning_rates, yticklabels=batch_sizes, cmap="Blues", ax=ax)
    ax.set_title(title)

plt.show()
